# NYC Green Taxi EDA in Pandas

This notebook loads NYC Green Taxi data, inspect it, clean columns, add date columns, enrich with holidays and weather, then create a final cleaned dataframe.


## 1. Setup

Run this first. If `azureml-opendatasets` is missing, install it in your VS Code terminal with:

```powershell
python -m pip install pandas azureml-opendatasets
```


In [ ]:
import pandas as pd
from datetime import datetime
from azureml.opendatasets import NycTlcGreen, PublicHolidays, NoaaIsdWeather

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)


## 2. Load NYC Green Taxi data

Use a small date range first while Exploring. A small range loads faster and is easier to inspect.


In [ ]:
start_date = datetime(2019, 1, 1)
end_date = datetime(2019, 6, 30)

nyc_tlc = NycTlcGreen(start_date=start_date, end_date=end_date)
nyc_tlc_df = nyc_tlc.to_pandas_dataframe()

nyc_tlc_df.head()


## 3. Basic EDA

In [ ]:
# Shape: rows and columns
rows, columns = nyc_tlc_df.shape
print(f"The dataset has {rows} rows and {columns} columns.")


In [ ]:
# Schema-like information: column names, data types, non-null counts
nyc_tlc_df.info()


In [ ]:
# First 5 rows in table format
nyc_tlc_df.head(5)


In [ ]:
# Column names
list(nyc_tlc_df.columns)


In [ ]:
# Statistical summary for all numeric columns
nyc_tlc_df.describe()


In [ ]:
# Statistical Summary for one field
nyc_tlc_df["fareAmount"].describe()


In [ ]:
# Missing value count for every column
missing_summary = (
    nyc_tlc_df
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

missing_summary["missing_percent"] = (missing_summary["missing_count"] / len(nyc_tlc_df) * 100).round(2)
missing_summary


## 4. Remove unused columns

`errors="ignore"` makes the drop safer. If one column name is missing or slightly different, the notebook will not fail.


In [ ]:
columns_to_remove = [
    "pickupLatitude",
    "pickupLongitude",
    "dropoffLatitude",
    "dropoffLongitude",
    "ehailFee"
]

nyc_tlc_df_clean = nyc_tlc_df.drop(columns=columns_to_remove, errors="ignore")
nyc_tlc_df_clean.head(5)


In [ ]:
# Random sample. Use min(5, row_count) so it will not fail for tiny datasets.
nyc_tlc_df_clean.sample(n=min(5, len(nyc_tlc_df_clean)), random_state=42)


In [ ]:
# Unique vendor IDs
nyc_tlc_df_clean["vendorID"].unique()


In [ ]:
# Count rows by paymentType
nyc_tlc_df_clean["paymentType"].value_counts(dropna=False)


In [ ]:
# Missing values for one column, only if the column exists
column_name = "pickupLongitude"

if column_name in nyc_tlc_df.columns:
    print(nyc_tlc_df[column_name].isna().sum())
else:
    print(f"Column not found: {column_name}")


## 5. Transformations

Create date/time fields from pickup datetime. 


In [ ]:
nyc_tlc_df_expand = nyc_tlc_df.assign(
    lpepPickupDatetime=pd.to_datetime(nyc_tlc_df["lpepPickupDatetime"]),
    datetime=lambda df: df["lpepPickupDatetime"].dt.date,
    month_num=lambda df: df["lpepPickupDatetime"].dt.month,
    day_of_month=lambda df: df["lpepPickupDatetime"].dt.day,
    day_of_week=lambda df: df["lpepPickupDatetime"].dt.dayofweek + 1,  # Monday=1, Sunday=7
    hour_of_day=lambda df: df["lpepPickupDatetime"].dt.hour,
    country_code="US",
)

nyc_tlc_df_expand.head()


## 6. Enrich with public holiday data

This loads holiday data and left-joins it to taxi data by `datetime` and `country_code`.


In [ ]:
hol = PublicHolidays(start_date=start_date, end_date=end_date)
hol_df = hol.to_pandas_dataframe()
hol_df.head()


In [ ]:
hol_df_clean = (
    hol_df
    .rename(columns={"countryRegionCode": "country_code"})
    .assign(datetime=lambda df: df["date"].dt.date)
    .drop(columns=["holidayName"], errors="ignore")
)

hol_df_clean.head()


In [ ]:
nyc_taxi_holiday_df = nyc_tlc_df_expand.merge(
    hol_df_clean,
    on=["datetime", "country_code"],
    how="left",
    suffixes=("", "_holiday"),
)

nyc_taxi_holiday_df.head()


In [ ]:
# Show only taxi rows that matched a holiday
holiday_rows = nyc_taxi_holiday_df[nyc_taxi_holiday_df["normalizeHolidayName"].notna()]
holiday_rows.head()


## 7. Enrich with NOAA weather data

This loads weather, filters roughly to NYC latitude/longitude, removes rows without temperature, and creates daily aggregates.


In [ ]:
isd = NoaaIsdWeather(start_date, end_date)
isd_df = isd.to_pandas_dataframe()
isd_df.head()


In [ ]:
weather_df = (
    isd_df
    .query("40.53 <= latitude <= 40.88 and -74.09 <= longitude <= -73.72")
    .dropna(subset=["temperature"])
    .rename(columns={"datetime": "datetime_full"})
)

weather_df.head()


In [ ]:
columns_to_remove_weather = ["usaf", "wban", "version","longitude", "latitude"]

weather_df_clean = (
    weather_df
    .drop(columns=columns_to_remove_weather, errors="ignore")
    .assign(datetime=lambda df:pd.to_datetime(df["datetime_full"]).dt.date)
)

weather_df_clean.head()


In [ ]:
weather_df_grouped = (
    weather_df_clean
    .groupby("datetime", as_index=False)
    .agg(
        avg_snowDepth=("snowDepth", "mean"),
        max_precipTime=("precipTime", "max"),
        avg_temperature=("temperature", "mean"),
        max_precipDepth=("precipDepth", "max"),
    )
)

weather_df_grouped.head()


## 8. Join taxi + holiday + weather


In [ ]:
nyc_taxi_holiday_weather_df = nyc_taxi_holiday_df.merge(
    weather_df_grouped,
    on="datetime",
    how="left",
)

nyc_taxi_holiday_weather_df.head()


In [ ]:
nyc_taxi_holiday_weather_df.describe()


## 9. Remove invalid rows

Removing rows where `tipAmount` or `totalAmount` are not positive.


In [ ]:
final_df = nyc_taxi_holiday_weather_df.query("tipAmount > 0 and totalAmount > 0").copy()

print(f"Before cleaning: {len(nyc_taxi_holiday_weather_df):,} rows")
print(f"After cleaning:  {len(final_df):,} rows")

final_df.head()


## 10. quick analysis examples


In [ ]:
# Average total amount by pickup hour
avg_by_hour = (
    final_df
    .groupby("hour_of_day", as_index=False)
    .agg(
        trips=("totalAmount", "size"),
        avg_total_amount=("totalAmount", "mean"),
        avg_tip_amount=("tipAmount", "mean"),
    )
    .sort_values("hour_of_day")
)

avg_by_hour


In [ ]:
# Top pickup days by trip count
trips_by_date = (
    final_df
    .groupby("datetime", as_index=False)
    .agg(trips=("totalAmount", "size"), avg_total_amount=("totalAmount", "mean"))
    .sort_values("trips", ascending=False)
)

trips_by_date.head(10)
